# SBERT Fine-tuning — NAICS Sector Prediction

**Before running:** Go to `Runtime > Change runtime type` and select **T4 GPU**.

You will need to upload three files when prompted:
- `sbert_logistic_classifier.py`
- `naics_metrics.py`
- `ExioNAICS_combined.csv`

In [ ]:
# Step 1 — Verify GPU
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu}  ({mem:.1f} GB)")
else:
    print("No GPU detected — go to Runtime > Change runtime type > T4 GPU and re-run")

In [ ]:
# Step 2 — Install dependencies
!pip install -q sentence-transformers transformers seaborn tqdm

In [ ]:
# Step 3 — Upload sbert_logistic_classifier.py and naics_metrics.py
from google.colab import files

print("Upload sbert_logistic_classifier.py")
files.upload()

print("\nUpload naics_metrics.py")
files.upload()

In [ ]:
# Step 4 — Upload data
import os
from google.colab import files

os.makedirs("/content/data", exist_ok=True)

print("Upload ExioNAICS_combined.csv")
uploaded = files.upload()

for filename, content in uploaded.items():
    dest = f"/content/data/{filename}"
    with open(dest, "wb") as f:
        f.write(content)
    print(f"Saved to {dest}")

In [ ]:
# Step 5 — Verify file structure
import os

required = [
    "/content/sbert_logistic_classifier.py",
    "/content/naics_metrics.py",
    "/content/data/ExioNAICS_combined.csv",
]

all_ok = True
for path in required:
    exists = os.path.exists(path)
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {path}")
    if not exists:
        all_ok = False

if all_ok:
    print("\nAll files found — ready to train.")
else:
    print("\nSome files are missing — re-run the upload cells above.")

In [ ]:
# Step 6 — Run fine-tuning (bge-base, pipeline validation)
# Expected time: ~5-15 min per epoch on T4
!python /content/sbert_logistic_classifier.py \
    --backbone BAAI/bge-base-en-v1.5 \
    --head torch_mlp \
    --hidden_dims 512 256 128 \
    --finetune \
    --mlp_batch_size 16 \
    --freeze_layers 4 \
    --epochs 10 \
    --max_length 96

In [ ]:
# Step 7 (optional) — Run with bge-large once pipeline is validated
# Expected time: ~5-10 min per epoch on T4
# !python /content/sbert_logistic_classifier.py \
#     --backbone BAAI/bge-large-en-v1.5 \
#     --head torch_mlp \
#     --hidden_dims 512 256 128 \
#     --finetune \
#     --mlp_batch_size 16 \
#     --freeze_layers 8 \
#     --epochs 10 \
#     --max_length 96

In [ ]:
# Step 8 — Download results
from google.colab import files
import os

output_dir = "/content/outputs"

if not os.path.exists(output_dir):
    print("No outputs directory found — training may not have completed.")
else:
    # Download top-level output files (predictions, plots)
    for fname in os.listdir(output_dir):
        fpath = os.path.join(output_dir, fname)
        if os.path.isfile(fpath):
            print(f"Downloading {fname}")
            files.download(fpath)

    # Download saved model artifacts
    model_dir = os.path.join(output_dir, "saved_model")
    if os.path.exists(model_dir):
        for fname in os.listdir(model_dir):
            fpath = os.path.join(model_dir, fname)
            if os.path.isfile(fpath):
                print(f"Downloading saved_model/{fname}")
                files.download(fpath)
    else:
        print("No saved_model directory found.")

In [ ]:
# Step 9 — Upload OOD dataset
from google.colab import files
import os

print("Upload ood_dataset_official.csv")
uploaded = files.upload()

for filename, content in uploaded.items():
    dest = f"/content/data/{filename}"
    with open(dest, "wb") as fh:
        fh.write(content)
    print(f"Saved to {dest}")

In [ ]:
# Step 10 — OOD evaluation
import json, os, sys
import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report

sys.path.insert(0, "/content")
from sbert_logistic_classifier import _SBERTWithMLP, TorchFinetuneClassifier, embed_texts
from naics_metrics import compute_naics_metrics, print_metrics as print_naics_metrics

OOD_CSV   = "/content/data/ood_dataset_official.csv"
MODEL_DIR = "/content/outputs/saved_model"

# Load config and label encoder
with open(f"{MODEL_DIR}/config.json") as f:
    cfg = json.load(f)
le = joblib.load(f"{MODEL_DIR}/label_encoder.joblib")

# Load and filter OOD data to known NAICS-2 classes
ood_df = pd.read_csv(OOD_CSV, dtype=str).dropna(subset=["model_input", "naics2_code"])
ood_df["model_input"] = ood_df["model_input"].str.strip()
ood_df["naics2_code"] = ood_df["naics2_code"].str.strip()

n_total = len(ood_df)
ood_df  = ood_df[ood_df["naics2_code"].isin(set(le.classes_))].reset_index(drop=True)
print(f"OOD rows: {n_total:,} total \u2192 {len(ood_df):,} with a known NAICS-2 class")

y_true = le.transform(ood_df["naics2_code"].tolist())
texts  = ood_df["model_input"].tolist()

# Load model and run inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if cfg["finetune"]:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = _SBERTWithMLP(cfg["backbone"], tuple(cfg["hidden_dims"]), cfg["num_classes"])
    model.load_state_dict(torch.load(f"{MODEL_DIR}/model_state.pt", map_location=device))
    model.to(device).eval()
    clf = TorchFinetuneClassifier(
        backbone_name=cfg["backbone"],
        num_classes=cfg["num_classes"],
        hidden_dims=tuple(cfg["hidden_dims"]),
        max_length=cfg["max_length"],
    )
    clf.model     = model
    clf.tokenizer = tokenizer
else:
    X_ood = embed_texts(texts, backbone=cfg["backbone"], batch_size=256)
    clf   = joblib.load(f"{MODEL_DIR}/classifier.joblib")
    texts = X_ood

y_pred_proba = clf.predict_proba(texts)
y_pred       = y_pred_proba.argmax(axis=1)

# Metrics
acc         = accuracy_score(y_true, y_pred)
macro_f1    = f1_score(y_true, y_pred, average="macro",    zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"\n{'='*60}")
print(f"  OOD TEST SET \u2014 NAICS{cfg['naics_level']} classification ({cfg['backbone']})")
print(f"{'='*60}")
print(f"  Accuracy    : {acc:.4f}")
print(f"  Macro-F1    : {macro_f1:.4f}   \u2190 primary metric")
print(f"  Weighted-F1 : {weighted_f1:.4f}")
print()

present_labels = sorted(set(y_true) | set(y_pred))
present_names  = le.inverse_transform(present_labels)
print(classification_report(y_true, y_pred, labels=present_labels,
                             target_names=present_names, zero_division=0))

ext = compute_naics_metrics(
    y_true       = ood_df["naics2_code"].tolist(),
    y_scores     = y_pred_proba,
    class_labels = le.classes_.tolist(),
)
print_naics_metrics(ext)

# Save OOD predictions
out = ood_df[["company name", "model_input", "naics2_code"]].copy()
out.columns = ["company_name", "text", "label"]
out["pred_label"] = le.inverse_transform(y_pred)
out["confidence"] = y_pred_proba.max(axis=1).round(4)
out["correct"]    = out["label"] == out["pred_label"]
pred_path = "/content/outputs/predictions_ood.csv"
os.makedirs("/content/outputs", exist_ok=True)
out.to_csv(pred_path, index=False)
print(f"\nOOD predictions saved \u2192 {pred_path}  ({len(out):,} rows)")